In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import requests

np.random.seed(123)

from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA

# 1. Simulación

In [ ]:
T = 100
media_y = 10
sigma = np.sqrt(3)

y1 = 9
y2 = 10.3

# ARMA

In [ ]:
def simular_arma(T, mu=10, phi=[], theta=[], sigma=np.sqrt(3), y_ini=[9, 10.3]):
    
    p = len(phi)
    q = len(theta)
    
    y = np.zeros(T)
    eps = np.random.normal(0, sigma, T)
    
    y[0] = y_ini[0]
    y[1] = y_ini[1]
    
    for t in range(2, T):
        
        ar = 0
        ma = 0
        
        for i in range(p):
            ar += phi[i] * y[t-i-1]
        
        for j in range(q):
            ma += theta[j] * eps[t-j-1]
        
        y[t] = mu + ar + eps[t] + ma
    
    return y

In [ ]:
series = {
    "AR(1) phi=0.4": simular_arma(T, mu=10, phi=[0.4]),
    "AR(1) phi=0.8": simular_arma(T, mu=10, phi=[0.8]),
    "AR(1) phi=0.99": simular_arma(T, mu=10, phi=[0.99]),
    
    "AR(2)": simular_arma(T, mu=10, phi=[0.4, -0.05]),
    
    "MA(1) theta=0.5": simular_arma(T, mu=10, theta=[0.5]),
    "MA(1) theta=2": simular_arma(T, mu=10, theta=[2]),
    
    "ARMA(1,1)": simular_arma(T, mu=10, phi=[0.4], theta=[0.5]),
    "ARMA(2,1)": simular_arma(T, mu=10, phi=[0.4, -0.2], theta=[0.5]),
    "ARMA(2,2)": simular_arma(T, mu=10, phi=[0.4, -0.2], theta=[0.5, 0.3])
}

In [ ]:
for nombre, y in series.items():
    plt.figure(figsize=(10,4))
    plt.plot(y)
    plt.title(nombre)
    plt.xlabel("Tiempo")
    plt.ylabel("y_t")
    plt.grid(True)
    plt.show()

# ARIMA(2,1,1)

In [ ]:
def simular_arima_211(T, phi=[0.4, -0.2], theta=[0.5], sigma=np.sqrt(3), y_ini=[9, 10.3]):
    
    dy = simular_arma(T, mu=0, phi=phi, theta=theta, sigma=sigma, y_ini=[0, 0])
    
    y = np.zeros(T)
    y[0] = y_ini[0]
    y[1] = y_ini[1]
    
    for t in range(2, T):
        y[t] = y[t-1] + dy[t]
    
    return y

arima_211 = simular_arima_211(T)

plt.figure(figsize=(10,4))
plt.plot(arima_211)
plt.title("ARIMA(2,1,1)")
plt.xlabel("Tiempo")
plt.ylabel("y_t")
plt.grid(True)
plt.show()

# 2. Máxima Verosimilitud ARMA(2,2)

In [ ]:
T = 100

mu = 10
phi1 = 0.4
phi2 = -0.2
theta1 = 0.5
theta2 = 0.3
sigma = np.sqrt(3)

y = np.zeros(T)
eps = np.random.normal(0, sigma, T)

y[0] = 9
y[1] = 10.3

for t in range(2, T):
    y[t] = (
        mu
        + phi1*y[t-1]
        + phi2*y[t-2]
        + eps[t]
        + theta1*eps[t-1]
        + theta2*eps[t-2]
    )

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(y)
plt.title("Serie simulada ARMA(2,2)")
plt.xlabel("Tiempo")
plt.ylabel("y_t")
plt.grid(True)
plt.show()

# Residuos

In [ ]:
def compute_residuals(psi, y):
    mu, phi1, phi2, theta1, theta2, sigma = psi
    
    T = len(y)
    e = np.zeros(T)
    
    e[0] = 0
    e[1] = 0
    
    for t in range(2, T):
        e[t] = (
            y[t]
            - mu
            - phi1*y[t-1]
            - phi2*y[t-2]
            - theta1*e[t-1]
            - theta2*e[t-2]
        )
    
    return e

# log-verosimilitud negativa

In [ ]:
def neg_log_likelihood(psi, y):
    mu, phi1, phi2, theta1, theta2, sigma = psi
    
    if sigma <= 0:
        return 1e10
    
    e = compute_residuals(psi, y)
    
    T = len(y)
    T_star = T - 2
    
    log_likelihood = (
        -(T_star/2)*np.log(2*np.pi*sigma**2)
        -(1/(2*sigma**2))*np.sum(e[2:]**2)
    )
    
    return -log_likelihood

# Estimación MLE propio

In [ ]:
psi0 = np.array([0, 0, 0, 0, 0, 1])

resultado_mle = minimize(
    neg_log_likelihood,
    psi0,
    args=(y,),
    method="L-BFGS-B",
    bounds=[
        (None, None),
        (-0.99, 0.99),
        (-0.99, 0.99),
        (-0.99, 0.99),
        (-0.99, 0.99),
        (1e-6, None)
    ]
)

psi_hat = resultado_mle.x
psi_hat

In [ ]:
tabla_mle = pd.DataFrame({
    "Parámetro": ["mu", "phi1", "phi2", "theta1", "theta2", "sigma"],
    "Verdadero": [mu, phi1, phi2, theta1, theta2, sigma],
    "MLE propio": psi_hat
})

tabla_mle

# Validación con statsmodels

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

modelo_sm = ARIMA(y, order=(2,0,2), trend="c")
resultado_sm = modelo_sm.fit()

resultado_sm.params

In [ ]:
params_sm = resultado_sm.params

tabla_final = pd.DataFrame({
    "Parámetro": ["mu", "phi1", "phi2", "theta1", "theta2", "sigma2"],
    "Verdadero": [mu, phi1, phi2, theta1, theta2, sigma**2],
    "MLE propio": [psi_hat[0], psi_hat[1], psi_hat[2], psi_hat[3], psi_hat[4], psi_hat[5]**2],
    "statsmodels": params_sm
})

tabla_final

Los resultados del MLE propio son cercanos a los parámetros verdaderos utilizados en la simulación, aunque no coinciden exactamente porque la muestra contiene solo 100 observaciones y existe ruido aleatorio. Al comparar con statsmodels, pueden aparecer pequeñas diferencias porque statsmodels utiliza una forma distinta de inicializar los residuos y maximizar la verosimilitud. Aun así, ambos métodos buscan el mismo objetivo, encontrar los parámetros que hacen más probable la serie observada.

# 3. ¿Qué puede decir respecto de la estacionariedad del IMACEC?

In [ ]:
import bcchapi

usuario = "antonia.cifuentes@usach.cl".strip()
clave = "Antonia01.".strip()

siete = bcchapi.Siete(usr=usuario, pwd=clave)

In [ ]:
codigo_imacec = "F032.IMC.IND.Z.Z.EP18.N03.Z.0.M"

imacec = siete.cuadro(
    series=[codigo_imacec],
    nombres=["IMACEC"],
    desde="2010-01-01",
    hasta="2024-12-31"
)

imacec.head()

In [ ]:
import pandas as pd

serie = imacec.copy()

serie.index = pd.to_datetime(serie.index)
serie["IMACEC"] = pd.to_numeric(serie["IMACEC"], errors="coerce")

serie = serie.dropna()

serie.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.plot(serie.index, serie["IMACEC"])
plt.title("IMACEC mensual (2010-2024)")
plt.xlabel("Fecha")
plt.ylabel("IMACEC")
plt.grid(True)
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

resultado = adfuller(serie["IMACEC"])

print("ADF:", resultado[0])
print("p-value:", resultado[1])

La serie del IMACEC no es estacionaria en niveles. La prueba ADF entregó un estadístico de -0.8353 y un p-value de 0.8085, por lo que no se rechaza la hipótesis nula de raíz unitaria. Esto indica que la serie presenta tendencia y requiere ser diferenciada antes de modelarla con un ARIMA.

In [ ]:
serie_diff = serie["IMACEC"].diff().dropna()

# Ljung-Box

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(serie["IMACEC"], lags=24)
plt.show()

plot_acf(serie_diff, lags=24)
plt.show()

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

print(acorr_ljungbox(serie["IMACEC"], lags=[24], return_df=True))
print(acorr_ljungbox(serie_diff, lags=[24], return_df=True))

La función de autocorrelación (ACF) del IMACEC muestra autocorrelaciones altas y persistentes durante los 24 rezagos, lo que indica que la serie tiene una fuerte dependencia temporal y no es estacionaria. Al aplicar la primera diferencia, las autocorrelaciones disminuyen considerablemente, lo que sugiere que la tendencia fue eliminada y que la serie es más adecuada para modelarla con un ARIMA.
El test de Ljung-Box confirma estos resultados. Para la serie original se obtuvo un estadístico de 1635.00 (p-value = 0.000) y para la primera diferencia un estadístico de 413.49 (p-value = 1.27 × 10⁻⁷²). En ambos casos se rechaza la hipótesis de ausencia de autocorrelación, lo que indica que la serie aún presenta dependencia temporal. Sin embargo, esta dependencia es menor después de diferenciar la serie, lo que justifica el uso de un modelo ARIMA para capturar esa estructura y realizar pronósticos.

# Box-Jenkins

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

modelos = {}

for p in range(3):
    for q in range(3):
        try:
            ajuste = ARIMA(serie["IMACEC"], order=(p,1,q)).fit()
            modelos[(p,1,q)] = ajuste.aic
        except:
            pass

import pandas as pd

tabla = (
    pd.DataFrame(modelos.items(), columns=["Modelo", "AIC"])
      .sort_values("AIC")
)

tabla

# Pronóstico

In [ ]:
modelo = ARIMA(serie["IMACEC"], order=(1,1,2)).fit()

pronostico = modelo.forecast(steps=3)

print(pronostico)

Al comparar estos resultados con la información oficial del Banco Central, se observa que el modelo proyecta una evolución positiva del IMACEC para los primeros meses de 2025. Si bien los valores no coinciden exactamente con los datos publicados por el Banco Central, esto es esperable porque el modelo ARIMA realiza las proyecciones únicamente a partir del comportamiento histórico del IMACEC y no incorpora otros factores económicos que pueden afectar la actividad económica, como cambios en la demanda, la política monetaria o eventos externos.
Es importante señalar que el modelo ARIMA entrega pronósticos del nivel del índice IMACEC, mientras que el Banco Central publica principalmente la variación porcentual anual del indicador. Por ello, la comparación se realiza de manera cualitativa, evaluando si el modelo reproduce la tendencia general observada en la actividad económica.

# 4. Inflación Chilena: Análisis Univariado de Series de Tiempo

## Datos desde la API del Banco Central

In [ ]:
codigos = [
    "G073.IPC.VAR.2023.M",
    "F022.TPM.TIN.D001.NO.Z.D",
    "F032.IMC.IND.Z.Z.EP18.N03.Z.0.M"
]

datos = siete.cuadro(
    series=codigos,
    nombres=["IPC_mensual", "TPM", "IMACEC_no_minero"],
    desde="2010-01-01",
    hasta="2024-12-31"
)

datos.index = pd.to_datetime(datos.index)

datos = datos.apply(pd.to_numeric, errors="coerce")

datos.head()

In [ ]:
datos_mensual = datos.resample("MS").mean()

datos_mensual.head()

In [ ]:
datos_mensual.tail()

# Inflación anual

In [ ]:
datos_mensual["IPC_anual"] = datos_mensual["IPC_mensual"].rolling(12).sum()

datos_mensual.head(15)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(datos_mensual.index, datos_mensual["IPC_mensual"], label="Inflación mensual")
plt.plot(datos_mensual.index, datos_mensual["IPC_anual"], label="Inflación anual acumulada 12 meses")

plt.axvspan("2020-03-01", "2020-12-31", alpha=0.2, label="COVID-19")
plt.axvspan("2021-01-01", "2022-12-31", alpha=0.2, label="Peak inflacionario")

plt.title("Inflación mensual y anual en Chile")
plt.xlabel("Fecha")
plt.ylabel("Variación (%)")
plt.legend()
plt.grid(True)
plt.show()

# Análisis de Estacionariedad

In [ ]:
ipc = datos_mensual["IPC_mensual"].dropna()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(ipc)
plt.title("Variación mensual del IPC")
plt.xlabel("Fecha")
plt.ylabel("%")
plt.grid(True)
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plot_acf(ipc, lags=24)
plt.show()

plot_pacf(ipc, lags=24)
plt.show()

Los gráficos muestran que la inflación mensual está relacionada con sus valores recientes, especialmente durante los primeros meses. Sin embargo, esa relación va disminuyendo con el tiempo, lo que sugiere que un modelo ARMA sencillo podría representar adecuadamente el comportamiento de la serie.

In [ ]:
from statsmodels.tsa.stattools import adfuller

adf = adfuller(ipc)

print("ADF:", adf[0])
print("p-value:", adf[1])

print("\nValores críticos")
for k,v in adf[4].items():
    print(k,":",v)

La prueba ADF entregó un estadístico de -3.1547 y un p-value de 0.0228. Como el p-value es menor a 0.05, se rechaza la hipótesis nula de raíz unitaria, concluyendo que la serie de inflación mensual es estacionaria y no requiere ser diferenciada antes de modelarla.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

des = seasonal_decompose(ipc,
                         model="additive",
                         period=12)

fig = des.plot()
fig.set_size_inches(12,8)
plt.show()

La descomposición muestra una componente estacional clara, ya que el patrón se repite de forma similar cada año. Además, se observa una tendencia creciente durante 2021-2022 y los residuos fluctúan alrededor de cero sin un patrón definido.

In [ ]:
from statsmodels.tsa.filters.hp_filter import hpfilter

ciclo, tendencia = hpfilter(ipc, lamb=14400)

plt.figure(figsize=(12,5))

plt.plot(ipc, label="Serie original")
plt.plot(tendencia, label="Tendencia")

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
ipc_desest = ipc - des.seasonal

ipc_desest.head()

# ARMA(p,q)

In [ ]:
ipc_modelo = ipc_desest.dropna()

ipc_modelo.head()

In [ ]:
ARIMA(ipc_modelo, order=(p,0,q))

# Tabla modelos

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

ipc_modelo = ipc_desest.dropna()

resultados = []

for p in range(4):
    for q in range(4):
        try:
            modelo = ARIMA(ipc_modelo, order=(p,0,q))
            ajuste = modelo.fit()
            
            resultados.append({
                "p": p,
                "q": q,
                "Modelo": f"ARMA({p},{q})",
                "AIC": ajuste.aic,
                "BIC": ajuste.bic,
                "HQIC": ajuste.hqic
            })
        except:
            pass

tabla_modelos = pd.DataFrame(resultados)

tabla_modelos = tabla_modelos.sort_values("BIC")

tabla_modelos

# Mejor modelo

In [ ]:
mejor_modelo = tabla_modelos.iloc[0]

mejor_modelo

In [ ]:
p_mejor = int(mejor_modelo["p"])
q_mejor = int(mejor_modelo["q"])

modelo_final = ARIMA(ipc_modelo, order=(p_mejor, 0, q_mejor))
resultado_final = modelo_final.fit()

print(resultado_final.summary())

Los criterios AIC y BIC no coinciden. El AIC selecciona el modelo ARMA(3,3), mientras que el BIC selecciona el ARMA(1,1). Esto ocurre porque el BIC penaliza con mayor fuerza la incorporación de parámetros adicionales, favoreciendo modelos más simples.
En este análisis prefiero el criterio BIC, ya que el objetivo es obtener un modelo que represente adecuadamente la dinámica de la inflación sin añadir complejidad innecesaria. Porque el modelo más sencillo suele ser más fácil de interpretar y reduce el riesgo de sobreajuste, especialmente cuando posteriormente se utilizará para realizar pronósticos.

# Coeficientes estimados

In [ ]:
tabla_coef = pd.DataFrame({
    "Coeficiente": resultado_final.params,
    "Error estándar": resultado_final.bse,
    "p-value": resultado_final.pvalues
})

tabla_coef

Todos los coeficientes son significativos, ya que sus p-values son menores a 0.05. Esto indica que cada uno aporta información importante para explicar el comportamiento de la inflación, por lo que el modelo ARMA(1,1) es adecuado para representar la serie.

# Diagnóstico del modelo

In [ ]:
residuos = resultado_final.resid

plt.figure(figsize=(12,4))
plt.plot(residuos)
plt.title("Residuos del modelo ARMA(1,1)")
plt.grid(True)
plt.show()

Los residuos oscilan alrededor de cero y no presentan un patrón evidente. Esto indica que el modelo ARMA(1,1) explica adecuadamente la dinámica de la serie y que la información restante en los residuos es principalmente aleatoria.

# ACF y PACF de los residuos

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(residuos, lags=24)
plt.show()

# Test de Ljung-Box

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

ljung = acorr_ljungbox(residuos, lags=[24], return_df=True)

ljung

Como el p-value (0.0591) es mayor a 0.05, no se rechaza la hipótesis nula del test de Ljung-Box. Esto indica que los residuos se comportan como ruido blanco, por lo que el modelo es adecuado.

# Test de Jarque-Bera

In [ ]:
from scipy.stats import jarque_bera

jb = jarque_bera(residuos)

print(jb)

Como el p-value (0.4059) es mayor a 0.05, no se rechaza la hipótesis de normalidad. Por lo tanto, los residuos presentan una distribución aproximadamente normal.

In [ ]:
plt.figure(figsize=(8,4))

plt.hist(residuos, bins=20)

plt.title("Distribución de los residuos")

plt.show()

El modelo pasa los principales diagnósticos. Los residuos no presentan un patrón evidente, el test de Ljung-Box indica que no existe autocorrelación significativa (p-value = 0.0591) y el test de Jarque-Bera muestra que los residuos se distribuyen aproximadamente de forma normal (p-value = 0.4059). Además, el histograma presenta una forma cercana a una distribución normal. Por lo tanto, el modelo ARMA(1,1) resulta adecuado para representar la inflación mensual desestacionalizada y realizar pronósticos.

En este caso no sería necesario realizar ajustes adicionales, ya que el modelo cumple satisfactoriamente con los diagnósticos realizados.

# Proyecciones

In [ ]:
pronostico = resultado_final.get_forecast(steps=3)

pred = pronostico.predicted_mean
ic = pronostico.conf_int()

print(pred)
ic

# Comparar los datos reales

In [ ]:
ipc_real = siete.cuadro(
    series=["G073.IPC.VAR.2023.M"],
    nombres=["IPC"],
    desde="2010-01-01",
    hasta="2025-03-31"
)

In [ ]:
ipc_2025 = ipc_real.loc["2025-01":"2025-03"]

ipc_2025

In [ ]:
tabla = pd.DataFrame({
    "Proyección": pred.values,
    "IC inf. 95%": ic.iloc[:,0].values,
    "IC sup. 95%": ic.iloc[:,1].values,
    "IPC efectivo": ipc_2025["IPC"].values
},
index=pred.index)

tabla

El modelo ARMA(1,1) generó pronósticos para enero, febrero y marzo de 2025 junto con intervalos de confianza al 95%. Al compararlos con los valores efectivos del IPC, se observa que las proyecciones fueron cercanas a los datos reales, aunque el modelo subestimó la inflación de enero. En febrero y marzo las diferencias fueron menores y los valores observados se mantuvieron dentro de los intervalos de confianza estimados, lo que indica un desempeño adecuado para pronósticos de corto plazo.

# RMSE y MAE

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np

mae = mean_absolute_error(
    tabla["IPC efectivo"],
    tabla["Proyección"]
)

rmse = np.sqrt(
    mean_squared_error(
        tabla["IPC efectivo"],
        tabla["Proyección"]
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

El error de las proyecciones fue cercano a 0.35 puntos porcentuales, mientras que el RMSE muestra un error ligeramente mayor al dar más peso a las diferencias más grandes. En general, ambas métricas sugieren que el modelo presenta un buen desempeño para realizar pronósticos de corto plazo.

# Limitaciones

Este modelo utiliza solo la información histórica de la inflación, por lo que no considera otros factores que pueden influir en su comportamiento. Para mejorar los pronósticos, sería útil incorporar variables como la TPM, el IMACEC o el tipo de cambio, ya que también afectan la evolución de la inflación.